# Práctica M57 - Visualización de Datos y Storytelling con Datos de Naciones Unidas

## Evolución de la población mundial y análisis demográfico mediante visualización efectiva

En esta práctica estoy desarrollando un análisis visual basado en datos oficiales de Naciones Unidas para contar una historia clara y significativa sobre la evolución demográfica mundial.

El objetivo es transformar datos complejos en una narrativa visual comprensible, aplicando los principios de visualización discutidos en el módulo (Gestalt, Tufte y Holmes).

## Importación de librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente")

## Carga y exploración inicial del dataset

In [ ]:
# Carga del dataset
df = pd.read_csv("population.csv", encoding="latin1", low_memory=False)

print("Columnas originales detectadas:")
print(df.columns.tolist())

# Ajuste de encabezados
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

# Renombrar columnas
df.columns = ["Area_Code", "Country", "Year", "Series", "Value", "Footnotes", "Source"]

# Conversión numérica
df["Value"] = df["Value"].astype(str).str.replace(",", "")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Limpieza de valores faltantes
df = df.dropna(subset=["Value", "Year"])

print("\nDataset cargado y limpiado correctamente.")
print("Dimensiones finales:", df.shape)

## Exploración inicial de los datos

In [ ]:
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nRango de años:", int(df["Year"].min()), "- ", int(df["Year"].max()))
print("\nIndicadores disponibles:")
print(df["Series"].value_counts().head(10))

## Selección de indicadores clave para la historia

In [ ]:
# Población total mundial
world_pop = df[
    (df["Country"] == "Total, all countries or areas") &
    (df["Series"] == "Population mid-year estimates (millions)")
].sort_values("Year")

# Población por sexo
male_pop = df[
    (df["Country"] == "Total, all countries or areas") &
    (df["Series"] == "Population mid-year estimates for males (millions)")
].sort_values("Year")

female_pop = df[
    (df["Country"] == "Total, all countries or areas") &
    (df["Series"] == "Population mid-year estimates for females (millions)")
].sort_values("Year")

# Densidad poblacional
density = df[
    (df["Country"] == "Total, all countries or areas") &
    (df["Series"] == "Population density")
].sort_values("Year")

print("Registros filtrados:")
print("Población total:", len(world_pop))
print("Población masculina:", len(male_pop))
print("Población femenina:", len(female_pop))
print("Densidad:", len(density))

## Visualización 1: Evolución de la población mundial

**Principio aplicado:** Claridad (Tufte) + Enfoque en los datos

In [ ]:
plt.figure(figsize=(11,6))
plt.plot(world_pop["Year"], world_pop["Value"]/1000, marker='o', markersize=5, linewidth=2.5, color='#1f77b4')

# Anotaciones clave
for year in [1950, 2000, 2020]:
    if year in world_pop["Year"].values:
        val = world_pop[world_pop["Year"] == year]["Value"].values[0] / 1000
        plt.annotate(f'{year}\n{val:.1f}B', xy=(year, val), xytext=(year, val + 0.8),
                     arrowprops=dict(arrowstyle='->', color='gray'), ha='center')

plt.axvline(x=2000, linestyle='--', alpha=0.4, color='gray')
plt.title("Crecimiento de la Población Mundial", fontsize=14, fontweight='bold')
plt.xlabel("Año")
plt.ylabel("Población (miles de millones)")
plt.grid(True, alpha=0.3)
plt.text(0.02, 0.95, 'Fuente: Naciones Unidas', transform=plt.gca().transAxes, fontsize=9, alpha=0.7)
plt.tight_layout()
plt.show()

**Storytelling:** La población mundial ha crecido de forma sostenida y acelerada, pasando de 2.5 mil millones en 1950 a casi 7.8 mil millones en 2020. Este crecimiento se intensificó notablemente desde la segunda mitad del siglo XX.

## Visualización 2: Comparación por sexo

**Principio aplicado:** Comparación (Gestalt) + Simplicidad (Tufte)

In [ ]:
sex_df = pd.merge(
    male_pop[["Year", "Value"]],
    female_pop[["Year", "Value"]],
    on="Year",
    suffixes=("_male", "_female")
)

plt.figure(figsize=(11,6))
plt.plot(sex_df["Year"], sex_df["Value_male"]/1000, label="Hombres", color='#1f77b4')
plt.plot(sex_df["Year"], sex_df["Value_female"]/1000, label="Mujeres", color='#ff7f0e')
plt.fill_between(sex_df["Year"], sex_df["Value_male"]/1000, sex_df["Value_female"]/1000, alpha=0.2, color='gray')
plt.title("Población Mundial por Sexo", fontsize=14, fontweight='bold')
plt.xlabel("Año")
plt.ylabel("Población (miles de millones)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.text(0.02, 0.95, 'Fuente: Naciones Unidas', transform=plt.gca().transAxes, fontsize=9, alpha=0.7)
plt.tight_layout()
plt.show()

**Storytelling:** Ambos sexos muestran una evolución demográfica muy similar a lo largo del tiempo, manteniendo un equilibrio poblacional con diferencias relativamente pequeñas.

## Visualización 3: Evolución de la densidad poblacional

**Principio aplicado:** Claridad y reducción de ruido (Tufte)

In [ ]:
plt.figure(figsize=(11,6))
plt.plot(density["Year"], density["Value"], marker='o', markersize=5, color='#2ca02c')

# Anotación final
last_year = density["Year"].iloc[-1]
last_val = density["Value"].iloc[-1]
plt.annotate(f'{last_val:.1f}', xy=(last_year, last_val), xytext=(last_year-15, last_val+5),
             arrowprops=dict(arrowstyle='->', color='gray'), ha='center')

plt.title("Evolución de la Densidad Poblacional Mundial", fontsize=14, fontweight='bold')
plt.xlabel("Año")
plt.ylabel("Habitantes por km²")
plt.grid(True, alpha=0.3)
plt.text(0.02, 0.95, 'Fuente: Naciones Unidas', transform=plt.gca().transAxes, fontsize=9, alpha=0.7)
plt.tight_layout()
plt.show()

## Hallazgos clave

- La población mundial se triplicó en menos de 70 años, pasando de 2.5 mil millones en 1950 a casi 7.8 mil millones en 2020.
- El crecimiento se aceleró notablemente a partir de la segunda mitad del siglo XX.
- La densidad poblacional aumentó de forma constante, reflejando mayor presión sobre el territorio.
- Se mantiene un equilibrio demográfico entre hombres y mujeres con diferencias mínimas.

## Storytelling Visual

La historia que cuentan estos datos es clara: la humanidad ha experimentado un crecimiento poblacional sin precedentes, pasando de 2.5 mil millones en 1950 a casi 7.8 mil millones en 2020, acompañado de un aumento continuo en la densidad. A pesar de ello, se mantiene un equilibrio aproximado entre hombres y mujeres.

Esta narrativa nos ayuda a reflexionar sobre los desafíos futuros en términos de recursos, urbanización y sostenibilidad.

## Principios de Visualización Aplicados

- **Principio Gestalt**: Agrupé elementos relacionados para facilitar la comparación visual.
- **Principios de Tufte**: Reduje elementos decorativos y maximicé la proporción de tinta dedicada a los datos.
- **Principios de Holmes**: Utilicé títulos claros y colores intuitivos para mejorar la comprensión del mensaje.

## Conclusión

El análisis evidencia que el crecimiento demográfico mundial ha sido sostenido y acelerado, pasando de 2.5 mil millones en 1950 a casi 7.8 mil millones en 2020, generando un aumento notable en la densidad de población.

La aplicación de principios de visualización efectiva me permitió transformar datos complejos en una historia clara y comprensible, cumpliendo con el objetivo de esta práctica.